# Project 3 - Credit Risk Modeling (Simplified Version)

Version allégée mais solide pour la soutenance.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
try:
    from imblearn.over_sampling import SMOTE
except Exception as exc:
    raise RuntimeError("Install imbalanced-learn: pip install -r requirements.txt") from exc

try:
    from xgboost import XGBClassifier
except Exception as exc:
    raise RuntimeError("XGBoost unavailable. On Mac: brew install libomp") from exc


In [ ]:
# Configuration
OUTPUT_DIR = Path('outputs')
CM_DIR = OUTPUT_DIR / 'confusion_matrices'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CM_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
THRESHOLD = 0.50

# Data loading: Colab upload or local fallback
try:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("No uploaded file.")
    DATA_PATH = Path(next(iter(uploaded.keys())))
    print(f"Data loaded via Colab upload: {DATA_PATH}")
except ImportError:
    DATA_PATH = Path('data/default_of_credit_card_clients.xls')
    print(f"Local fallback: {DATA_PATH}")


In [ ]:
def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(' ', '_')
        .replace('.', '_')
        .replace('/', '_')
        .replace('-', '_')
    )


def load_credit_dataset(data_path: Path) -> pd.DataFrame:
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found: {data_path}")

    if data_path.suffix.lower() == '.csv':
        df = pd.read_csv(data_path)
    elif data_path.suffix.lower() in {'.xls', '.xlsx'}:
        try:
            df = pd.read_excel(data_path)
            if 'default.payment.next.month' not in [str(c) for c in df.columns]:
                df = pd.read_excel(data_path, header=1)
        except Exception:
            df = pd.read_excel(data_path, header=1)
    else:
        raise ValueError('Unsupported format. Use .csv/.xls/.xlsx')

    df = df.loc[:, ~df.columns.astype(str).str.contains('^Unnamed')]
    df.columns = [normalize_column_name(c) for c in df.columns]
    return df


def detect_target_column(columns: list[str]) -> str:
    for c in ['default_payment_next_month', 'default', 'y', 'target']:
        if c in columns:
            return c
    raise ValueError('Target not found.')


def add_simple_features(df: pd.DataFrame):
    df = df.copy()
    created = []

    bill_cols = [f'bill_amt{i}' for i in range(1, 7) if f'bill_amt{i}' in df.columns]
    pay_cols = [f'pay_amt{i}' for i in range(1, 7) if f'pay_amt{i}' in df.columns]
    pay_status_cols = [c for c in ['pay_0','pay_2','pay_3','pay_4','pay_5','pay_6'] if c in df.columns]

    if 'limit_bal' in df.columns and len(bill_cols) > 0:
        df['util_ratio_mean'] = (df[bill_cols].abs().mean(axis=1) / (df['limit_bal'].replace(0, np.nan))).fillna(0)
        created.append('util_ratio_mean')

    if len(bill_cols) > 0 and len(pay_cols) > 0:
        df['pay_to_bill_total_ratio'] = df[pay_cols].sum(axis=1) / (df[bill_cols].abs().sum(axis=1) + 1)
        created.append('pay_to_bill_total_ratio')

    if len(pay_status_cols) > 0:
        df['delinquency_count'] = (df[pay_status_cols] > 0).sum(axis=1)
        created.append('delinquency_count')

    return df, created


## 1) Data loading + quick checks

In [ ]:
df = load_credit_dataset(DATA_PATH)
target_col = detect_target_column(list(df.columns))

id_cols = [c for c in df.columns if c in {'id', 'id_'} or c.startswith('id_')]
if id_cols:
    df = df.drop(columns=id_cols)

df, created_features = add_simple_features(df)

print('Shape:', df.shape)
print('Target:', target_col)
print('Created features:', created_features)

display(df.head())
print('\nTarget ratio:')
display(df[target_col].value_counts(normalize=True).rename('ratio'))

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('\nMissing values:')
if len(missing) == 0:
    print('No missing values')
else:
    display(missing)


## 2) EDA (concise)

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x=df[target_col])
plt.title('Target distribution')
plt.tight_layout()
plt.show()

plot_cols = [c for c in ['limit_bal', 'age', 'util_ratio_mean', 'pay_to_bill_total_ratio', 'delinquency_count'] if c in df.columns]
if len(plot_cols) > 0:
    fig, axes = plt.subplots(1, len(plot_cols), figsize=(4*len(plot_cols), 3))
    if len(plot_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, plot_cols):
        sns.histplot(df[col], bins=30, ax=ax)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

num_df = df.select_dtypes(include=[np.number])
if target_col in num_df.columns:
    corr = num_df.corr(numeric_only=True)[target_col].drop(target_col).sort_values(key=lambda s: s.abs(), ascending=False)
    display(corr.head(10).to_frame('corr_with_target'))


## 3) Split + class imbalance setup

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / max(pos, 1)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('scale_pos_weight:', round(scale_pos_weight, 2))

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print('After SMOTE:', pd.Series(y_train_smote).value_counts().to_dict())


## 4) Required models

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1200, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=220,
        max_depth=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='auc',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
    'Neural Network (MLP)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=250, early_stopping=True, random_state=RANDOM_STATE))
    ])
}

print(list(models.keys()))


## 5) Quick CV (optional but fast)

In [ ]:
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)

sample_n = min(8000, len(X_train))
X_cv = X_train.sample(n=sample_n, random_state=RANDOM_STATE)
y_cv = y_train.loc[X_cv.index]

cv_models = {
    'Logistic Regression': clone(models['Logistic Regression']),
    'Random Forest': clone(models['Random Forest']),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='auc',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
}

cv_rows = []
for name, model in cv_models.items():
    scores = cross_validate(
        model,
        X_cv,
        y_cv,
        cv=cv,
        scoring={'auc': 'roc_auc', 'f1': 'f1'},
        n_jobs=-1,
    )
    cv_rows.append({
        'model': name,
        'cv_auc_mean': round(float(np.mean(scores['test_auc'])), 4),
        'cv_f1_mean': round(float(np.mean(scores['test_f1'])), 4),
    })

cv_df = pd.DataFrame(cv_rows).sort_values('cv_auc_mean', ascending=False)
display(cv_df)
cv_df.to_csv(OUTPUT_DIR / 'cv_results.csv', index=False)


## 6) Final training + evaluation

In [ ]:
results = []
proba_map = {}
importance_data = []

plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], '--', label='random')

for name, model in models.items():
    if name == 'Neural Network (MLP)':
        model.fit(X_train_smote, y_train_smote)
    else:
        model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= THRESHOLD).astype(int)
    proba_map[name] = y_prob

    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        'model': name,
        'auc': round(float(auc), 4),
        'f1_score': round(float(f1), 4),
        'precision': round(float(prec), 4),
        'recall': round(float(rec), 4),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    })

    fig, ax = plt.subplots(figsize=(4.8, 4))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1]).plot(ax=ax, colorbar=False)
    ax.set_title(f'CM - {name}')
    plt.tight_layout()
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    fig.savefig(CM_DIR / f'cm_{safe_name}.png')
    plt.close(fig)

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

    if name == 'Random Forest' and hasattr(model, 'feature_importances_'):
        for f, v in zip(X_train.columns, model.feature_importances_):
            importance_data.append({'model': 'Random Forest', 'feature': f, 'importance': float(v)})
    if name == 'XGBoost' and hasattr(model, 'feature_importances_'):
        for f, v in zip(X_train.columns, model.feature_importances_):
            importance_data.append({'model': 'XGBoost', 'feature': f, 'importance': float(v)})

# Logistic coefficients
log_coef = models['Logistic Regression'].named_steps['model'].coef_[0]
for f, v in zip(X_train.columns, np.abs(log_coef)):
    importance_data.append({'model': 'Logistic |coef|', 'feature': f, 'importance': float(v)})

plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC curves')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curves.png')
plt.show()

results_df = pd.DataFrame(results).sort_values('auc', ascending=False)
if 'cv_df' in globals() and len(cv_df) > 0:
    final_df = results_df.merge(cv_df, on='model', how='left')
else:
    final_df = results_df.copy()

display(final_df)

importance_df = pd.DataFrame(importance_data)
for m in importance_df['model'].unique():
    print(f'\nTop 8 features - {m}')
    display(importance_df[importance_df['model'] == m].sort_values('importance', ascending=False).head(8))

final_df.to_csv(OUTPUT_DIR / 'model_results.csv', index=False)
importance_df.to_csv(OUTPUT_DIR / 'feature_importance_summary.csv', index=False)


## 7) Threshold trade-off (simple)

In [ ]:
best_model = final_df.sort_values('auc', ascending=False).iloc[0]['model']
probs = proba_map[best_model]

rows = []
for t in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (probs >= t).astype(int)
    cm = confusion_matrix(y_test, pred)
    rows.append({
        'model': best_model,
        'threshold': t,
        'f1': round(float(f1_score(y_test, pred)), 4),
        'precision': round(float(precision_score(y_test, pred, zero_division=0)), 4),
        'recall': round(float(recall_score(y_test, pred, zero_division=0)), 4),
        'tn': int(cm[0, 0]), 'fp': int(cm[0, 1]), 'fn': int(cm[1, 0]), 'tp': int(cm[1, 1]),
    })

thr_df = pd.DataFrame(rows).sort_values('f1', ascending=False)
display(thr_df)

recommended_threshold = float(thr_df.iloc[0]['threshold'])
print(f'Recommended threshold: {recommended_threshold:.2f}')

thr_df.to_csv(OUTPUT_DIR / 'threshold_analysis_best_model.csv', index=False)


## 8) Business notes (concise)

In [ ]:
notes = []
notes.append(f"Default rate = {float(df[target_col].mean()):.1%} (imbalanced problem).")
notes.append("Higher delinquency and lower payment-to-bill ratios are associated with higher risk.")
notes.append(f"Best model by AUC: {final_df.sort_values('auc', ascending=False).iloc[0]['model']}.")
notes.append(f"Threshold recommendation: {recommended_threshold:.2f} to balance precision/recall.")
notes.append("Light CV was used to keep runtime manageable and avoid over-complication.")

for i, n in enumerate(notes, 1):
    print(f"{i}. {n}")

pd.DataFrame({'business_note': notes}).to_csv(OUTPUT_DIR / 'business_observations_for_slides.csv', index=False)


## Outputs to use in slides
- `outputs/model_results.csv`
- `outputs/cv_results.csv`
- `outputs/threshold_analysis_best_model.csv`
- `outputs/feature_importance_summary.csv`
- `outputs/roc_curves.png`
- `outputs/confusion_matrices/*.png`
